In [1]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
import pinecone
from sentence_transformers import SentenceTransformer

/opt/anaconda3/envs/langchain_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# import chardet

# with open("course_descriptions.csv", "rb") as f:
#     result = chardet.detect(f.read())

# print(result)

In [3]:
files = pd.read_csv("course_descriptions.csv", encoding="windows-1252")

In [4]:
files.head()

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a..."
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...


In [5]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is {row["course_slug"]},
            the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [6]:
files["course_description_new"]=files.apply(create_course_description,axis=1)

In [7]:
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
                             ...                        
101    The course name is Intro to NLP for AI, the sl...
102    The course name is Data Analysis with ChatGPT,...
103    The course name is ChatGPT for Data Science, t...
104    The course name is Intro to LLMs, the slug is ...
105    The course name is Growth Analysis with SQL, P...
Name: course_description_new, Length: 106, dtype: object


In [8]:
%load_ext dotenv
%dotenv

In [9]:
load_dotenv(find_dotenv(), override = True)

True

In [10]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY"))

In [11]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [12]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

my-index succesfully deleted.


In [13]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

IndexModel(name='my-index', metric='cosine', host='https://my-index-sb80rb8.svc.aped-4627-b74a.pinecone.io', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), vector_type='dense', dimension=384, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [14]:
index = pc.Index(index_name)

### Embedding the data

In [15]:
model = SentenceTransformer("all-MiniLM-L6-v2")
# model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 10337.23it/s]


In [16]:
def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ['course_description', 'course_description_new', 'course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar = False)
    return embedding

In [17]:
files["embedding"] = files.apply(create_embeddings, axis = 1)

In [18]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors = vectors_to_upsert)

print("Data upserted to Pinecone index")

Data upserted to Pinecone index


### Semantic search

In [19]:
query="clustering"
query_embedding= model.encode(query, show_progress_bar = False).tolist()

In [20]:
query_result=index.query(vector=[query_embedding],top_k=12,include_values=True)
query_result

QueryResponse(matches=[ScoredVector(id='Machine Learning in Excel', score=0.354952842, values=[-0.0183002688, -0.0279485583, -0.025320366, ...381 more]), ScoredVector(id='Machine Learning with K-Nearest Neighbors', score=0.314138889, values=[-0.102056853, -0.0829932392, 0.0194283351, ...381 more]), ScoredVector(id='Machine Learning in Python', score=0.282951355, values=[-0.0447243899, -0.0423180163, -0.00483987, ...381 more]), ScoredVector(id='Customer Churn Analysis with SQL and Tableau', score=0.281317711, values=[-0.025576137, -0.0562176183, -0.0636670142, ...381 more]), ScoredVector(id='Growth Analysis with SQL, Python, and Tableau  ', score=0.259746552, values=[0.0178535637, -0.029345369, -0.0790975913, ...381 more]), ScoredVector(id='Linear Algebra and Feature Selection', score=0.25903511, values=[-0.0354893804, -0.0207994692, 0.0170970242, ...381 more]), ScoredVector(id='Customer Engagement Analysis with SQL and Tableau', score=0.234294906, values=[0.0382171758, -0.0132275699, -

In [28]:
score_threshold=0.3
for match in query_result["matches"]:
    if match['score']>=0.3:
        print(f"MAtched item ID:{match['id']},score:{match['score']}")

MAtched item ID:Machine Learning in Excel,score:0.354952842
MAtched item ID:Machine Learning with K-Nearest Neighbors,score:0.314138889
